# COHA Word2Vec — Interactive Global Trajectory

COHA Word2Vec — Interactive Global Trajectory (Plotly + Widget)
===============================================================
This notebook produces ONE interactive Plotly visualisation:
  "Évolution sémantique de <word> entre sous-corpus"

The plot shows:
  - A grey background word cloud  (global vocabulary in 2D PCA space)
  - A coloured trajectory line    (where the query word sits per decade)
  - A red X reference point       (where the word sits in the global model)

At the bottom, an interactive widget panel lets you change the query word,
adjust parameters, and re-run all outputs without re-training anything.

Pipeline
--------
1.  Load COHA .txt files — year extracted from filename → grouped by decade
2.  Clean text — strip COHA markup, lemmatise, remove stopwords (spaCy)
3.  Sentence-tokenise each decade
4.  Train one Word2Vec model per decade
5.  Train one global Word2Vec model on ALL decades combined
6.  Fit PCA on the global model vocabulary — this is the fixed background
7.  Align every decade model to the global model (Procrustes rotation)
8.  Project aligned decade vectors into the global PCA space
9.  Plot interactively with Plotly
10. Widget panel — change word, topn, background size without re-training

Requirements
------------
    pip install gensim spacy scikit-learn scipy plotly ipywidgets matplotlib
    python -m spacy download en_core_web_sm


In [ ]:
import re
import glob
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import spacy
!pip install gensim
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from scipy.linalg import orthogonal_procrustes
from scipy.spatial.distance import cosine as cosine_dist

# CONFIGURE
QUERY_WORD   = "love"            # starting word for the explorer
COHA_GLOB    = "/content/*.txt"
N_BACKGROUND = 3000              # how many global vocab words in background

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.1 MB/s eta 0:00:00


In [ ]:
# Extract decade from filename
# COHA filenames follow the pattern  fic_YEAR_NNNN.txt
#fic_1847_7650.txt  →  year = 1847  →  decade = 1840
#The regex captures the four-digit year,and integer division rounds it down to the nearest decade.

def extract_decade(filename):
    """Return the decade (e.g. 1840) from a COHA filename, or None."""
    match = re.search(r'fic_(\d{4})_', filename)
    if match:
        year   = int(match.group(1))
        decade = (year // 10) * 10   # 1847 → 1840
        return decade
    return None


In [ ]:
import os
# Load raw documents grouped by decade
# Each COHA file can contain multiple documents separated by @@NUMBER markers.
# split on those markers so every segment becomes its own document string.
# Result: raw_docs_by_decade = {1830: ["text...", ...], 1840: [...], …}

file_paths = sorted(glob.glob(COHA_GLOB))
if not file_paths:
    raise FileNotFoundError(
        f"No .txt files found at {COHA_GLOB!r}.\n"
        "Update the COHA_GLOB path in the configuration block above."
    )

raw_docs_by_decade = defaultdict(list)

for fp in file_paths:
    decade = extract_decade(os.path.basename(fp))
    if decade is None:
        continue

    with open(fp, encoding="utf-8", errors="replace") as f:
        content = f.read()

    # @@1234 marks document boundaries inside a COHA file
    for part in re.split(r"@@\d+", content):
        stripped = part.strip()
        if stripped:
            raw_docs_by_decade[decade].append(stripped)

print("Decades loaded:", sorted(raw_docs_by_decade.keys()))
for dec in sorted(raw_docs_by_decade.keys()):
    print(f"  {dec}s → {len(raw_docs_by_decade[dec])} documents")

Decades loaded: [1810, 1820, 1830, 1840, 1860, 1870, 1880, 1890, 1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000]
  1810s → 1 documents
  1820s → 1 documents
  1830s → 5 documents
  1840s → 2 documents
  1860s → 1 documents
  1870s → 4 documents
  1880s → 4 documents
  1890s → 4 documents
  1900s → 2 documents
  1910s → 4 documents
  1920s → 1 documents
  1930s → 6 documents
  1940s → 7 documents
  1950s → 3 documents
  1960s → 5 documents
  1970s → 3 documents
  1980s → 2 documents
  1990s → 14 documents
  2000s → 39 documents


## Text cleaning


In [ ]:
# Text cleaning
# Two-pass pipeline:
#   Pass A — coha_preprocess()      : remove COHA-specific noise
#   Pass B — spacy_clean_sentence() : lemmatise and remove stopwords
#
#use spaCy's sentencizer rule-based, fast)

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp.add_pipe("sentencizer")


def coha_preprocess(text: str) -> str:
    """
    Remove COHA markup and dramatic-text noise.

    Removes: redaction markers (@ @ @ @), parenthetical stage directions,
    structural headers (ACT II, SCENE III …), character name prefixes,
    and ALL-CAPS stage labels.
    Punctuation is kept here so the sentencizer can detect sentence ends.
    """
    text = re.sub(r"(?:@\s*)+", " ", text)                    # redaction marks
    text = re.sub(r"\([^)]*\)", " ", text)                    # stage directions
    for h in [r"Main text", r"DRAMATIS PERSON\.?",
              r"ACT\s+[IVXLC]+\.?", r"SCENE\s+[IVXLC]+",
              r"SCENE\s+\w+", r"CHORUS", r"END\s+OF\s+ACT",
              r"STAGE DIRECTIONS", r"EXITS? AND ENTRANCES?",
              r"RELATIVE POSITIONS?"]:
        text = re.sub(h, " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[A-Z][a-z]{0,6}\s*\.\s*[A-Z][a-zA-Z]+\s+", " ", text)
    text = re.sub(r"\b[A-Z]{2,}\b\.?", " ", text)            # ALL-CAPS labels
    return re.sub(r"\s+", " ", text).strip()


def spacy_clean_sentence(span) -> list[str] | None:
    """
    Lemmatise one spaCy sentence span.
    Returns a list of lowercase lemma strings, or None if fewer than
    3 tokens remain after removing stopwords and punctuation.

    Lemmatisation maps "loved", "loves", "loving" → "love" so all
    forms contribute to the same word vector — important for small corpora.
    """
    tokens = [
        t.lemma_.lower()
        for t in span
        if not t.is_stop and t.is_alpha and len(t.text) > 1
    ]
    return tokens if len(tokens) > 2 else None


## Sentence-tokenise each decade


In [ ]:
# Result: sentences_by_decade = {1830: [[token,…], [token,…], …], 1840: …}
# Each inner list is one cleaned, tokenised sentence ready for Word2Vec.

sentences_by_decade = {}

for decade, docs in raw_docs_by_decade.items():
    prepped          = [coha_preprocess(doc) for doc in docs]
    decade_sentences = []

    for spacy_doc in nlp.pipe(prepped, batch_size=200):
        for sent in spacy_doc.sents:
            cleaned = spacy_clean_sentence(sent)
            if cleaned:
                decade_sentences.append(cleaned)

    sentences_by_decade[decade] = decade_sentences
    print(f"  {decade}s → {len(decade_sentences)} sentences")


  1810s → 665 sentences
  1820s → 332 sentences
  1830s → 4981 sentences
  1840s → 733 sentences
  1860s → 265 sentences
  1870s → 8021 sentences
  1880s → 9571 sentences
  1890s → 6248 sentences
  1900s → 4503 sentences
  1910s → 6107 sentences
  1920s → 4950 sentences
  1930s → 5957 sentences
  1940s → 7342 sentences
  1950s → 2343 sentences
  1960s → 6933 sentences
  1970s → 10027 sentences
  1980s → 2654 sentences
  1990s → 2499 sentences
  2000s → 6482 sentences


## Train one Word2Vec model per decade


In [ ]:
#Each model only sees its own decade's text, which is what gives us the
# each model captures how words were used in that period.
#
# Parameters for a small-to-medium COHA sample:
#   vector_size = 100  — raise to 300 with the full COHA corpus
#   window      = 5    — context window: ±5 words
#   min_count   = 3    — ignore words appearing fewer than 3 times
#   epochs      = 30   — 30 full passes through the sentences

models = {}

for decade, sentences in sentences_by_decade.items():
    if not sentences:
        print(f"  {decade}s — no sentences, skipping.")
        continue
    print(f"  Training {decade}s …", end=" ")
    model = Word2Vec(
        sentences   = sentences,
        vector_size = 100,
        window      = 5,
        min_count   = 3,
        workers     = 2,
        epochs      = 30,
        seed        = 42,
    )
    models[decade] = model
    print(f"vocab = {len(model.wv)} words")


## Train the global model


In [ ]:
# Flatten ALL decades into one big sentence list and train a single model.
#
# The global model sees the whole corpus at once.
# Its vector space becomes the reference that decade models will be rotated into w/ Procrustes alignment
# also the background word cloud and the red X reference point where the query word sits approx

all_sentences = [
    sentence
    for decade_sentences in sentences_by_decade.values()
    for sentence in decade_sentences
]

print(f"\nTraining global model on {len(all_sentences)} sentences …")
global_model = Word2Vec(
    sentences   = all_sentences,
    vector_size = 100,   # must match per-decade vector_size
    window      = 5,
    min_count   = 3,
    workers     = 2,
    epochs      = 30,
    seed        = 42,
)
print(f"Global model vocabulary: {len(global_model.wv)} words")


## Fit the global PCA — the fixed coordinate system


In [ ]:
# Take the N_BACKGROUND most frequent words from the global model,
# get their 100-dimensional vectors, and fit PCA to reduce to 2D.
#
# This PCA object (pca_global) is fitted ONCE and never re-fitted.
# All decade positions are later projected with pca_global.transform()
# (not fit_transform) so they land in the same fixed 2D space as the
# background words.
#
# The axes show how much variance each dimension explains — typically
# Axe 1 captures the broadest semantic distinction in the corpus.

global_wv = global_model.wv

# sort words by frequency most common first and take top N_BACKGROUND
all_global_words = sorted(
    global_wv.key_to_index.keys(),
    key     = lambda w: global_wv.get_vecattr(w, "count"),
    reverse = True,
)[:N_BACKGROUND]

global_bg_vectors = np.array([global_wv[w] for w in all_global_words])

pca_global       = PCA(n_components=2, random_state=42)
global_bg_coords = pca_global.fit_transform(global_bg_vectors)

var1 = pca_global.explained_variance_ratio_[0] * 100
var2 = pca_global.explained_variance_ratio_[1] * 100
print(f"\nGlobal PCA: {len(all_global_words)} words")
print(f"Variance explained — Axe 1: {var1:.1f}%   Axe 2: {var2:.1f}%")


## Procrustes alignment — rotate every decade into the global space


In [ ]:
# each Word2Vec model trains with a random starting orientation.

# Procrustes alignment for each decade model it finds the
# rotation matrix R such that the words present in both the decade model and the global model land as close as possible
# to their positions in the global model.
# Solved in one step using SVD
#
# After alignment  current_vecs @ R  is in the global coordinate system
# so  pca_global.transform()  can project it correctly onto the background.

def align_all_to_global(models, global_model):
    """
    Rotate every decade model into the global model's coordinate system.

    Returns {decade: {word: aligned_vector (numpy array)}}
    Only words shared between the decade model and global model are kept —
    these are the anchor words used to compute the rotation.
    """
    global_wv = global_model.wv
    aligned   = {}

    for decade, model in models.items():
        current_wv = model.wv

        # Anchor set — words present in BOTH this decade AND global model
        # More anchors = more stable, reliable rotation
        common = list(
            set(global_wv.key_to_index) & set(current_wv.key_to_index)
        )

        if len(common) < 2:
            print(f"  Skipping {decade}s — fewer than 2 shared words")
            continue

        # Build two matrices with the same row order in both:
        #   row 0 = "heart" in global model    row 0 = "heart" in 1830s model
        #   row 1 = "love"  in global model    row 1 = "love"  in 1830s model
        global_vecs  = np.array([global_wv[w]  for w in common])
        current_vecs = np.array([current_wv[w] for w in common])

        # Find R  →  current_vecs @ R  ≈  global_vecs
        R, _         = orthogonal_procrustes(current_vecs, global_vecs)
        aligned_vecs = current_vecs @ R   # apply the rotation

        aligned[decade] = {w: aligned_vecs[i] for i, w in enumerate(common)}
        print(f"  {decade}s aligned — {len(common)} anchor words")

    return aligned


print("\nAligning decade models to global model …")
aligned_to_global = align_all_to_global(models, global_model)
print(f"Alignment complete — {len(aligned_to_global)} decades ready\n")


## Helper — project decade positions into global PCA space


In [ ]:
# Each aligned decade vector is now in the global model's coordinate system.
# We call pca_global.transform() not fit_transform — to project into the
# same 2D plane as the background without moving any existing axes.

def get_decade_positions(aligned_to_global, query_word, pca_global):
    """
    Return (decade, x, y) for every decade containing query_word.
    Coordinates are in the global PCA space — comparable to background words.
    """
    positions = []
    for decade in sorted(aligned_to_global.keys()):
        vecs = aligned_to_global[decade]
        if query_word not in vecs:
            continue
        # reshape(1,-1) converts 1-D vector to 2-D row — required by transform
        coords = pca_global.transform(vecs[query_word].reshape(1, -1))[0]
        positions.append((decade, float(coords[0]), float(coords[1])))
    return positions


## cosine similarity between two vectors


In [ ]:
# Used by the analysis functions below.

def cosine_sim(v1, v2):
    """Cosine similarity between two 1-D numpy vectors. Returns float in [-1,1]."""
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (n1 * n2))


## nearest neighbours per decade


In [ ]:
# For each decade, finds the topn words most similar to query_word
# using the ALIGNED vectors so scores are comparable across decades.
# Also retrieves raw frequency and frequency rank from the original model.

def nearest_neighbours_per_decade(aligned_to_global, models, query_word,
                                   topn=10, min_freq=1):
    """
    Returns {decade: DataFrame} with columns:
        word | similarity | frequency | freq_rank

    Uses aligned vectors for similarity (comparable across decades).
    Uses original model for frequency info.
    """
    results = {}

    for decade in sorted(aligned_to_global.keys()):
        vecs = aligned_to_global[decade]
        if query_word not in vecs:
            continue

        qvec = vecs[query_word]

        # Score every word in this aligned decade against the query word
        sims = [
            (word, cosine_sim(qvec, vec))
            for word, vec in vecs.items()
            if word != query_word
        ]
        sims.sort(key=lambda x: x[1], reverse=True)

        # Add frequency info from the original (unaligned) decade model
        wv = models[decade].wv
        freq_rank = {
            w: i + 1
            for i, w in enumerate(
                sorted(wv.key_to_index, key=lambda w: wv.get_vecattr(w, "count"),
                       reverse=True)
            )
        }

        rows = []
        for word, sim in sims[:topn * 3]:   # over-fetch then filter
            if word not in wv:
                continue
            freq = int(wv.get_vecattr(word, "count"))
            if freq < min_freq:
                continue
            rows.append({
                "word":      word,
                "similarity": round(sim, 4),
                "frequency":  freq,
                "freq_rank":  freq_rank.get(word, -1),
            })
            if len(rows) >= topn:
                break

        if rows:
            results[decade] = pd.DataFrame(rows)

    return results


## distance traveled per decade step


In [ ]:
# Computes cosine and Euclidean distance between consecutive decade positions.
# A large distance = big semantic shift in that step.
# Uses aligned vectors so distances are comparable across decades.

def compute_trajectory_distances(aligned_to_global, query_word):
    """
    Returns a DataFrame with one row per consecutive decade transition:
        step | from_decade | to_decade | cosine_dist | euclidean_dist
    """
    decades_with_word = [
        d for d in sorted(aligned_to_global.keys())
        if query_word in aligned_to_global[d]
    ]

    rows = []
    for i in range(len(decades_with_word) - 1):
        d1 = decades_with_word[i]
        d2 = decades_with_word[i + 1]
        v1 = aligned_to_global[d1][query_word]
        v2 = aligned_to_global[d2][query_word]
        rows.append({
            "step":           f"{d1}s → {d2}s",
            "from_decade":    d1,
            "to_decade":      d2,
            "cosine_dist":    round(float(cosine_dist(v1, v2)), 4),
            "euclidean_dist": round(float(np.linalg.norm(v1 - v2)), 4),
        })

    return pd.DataFrame(rows)


## most semantically dispersed words


In [ ]:
# For every word present in at least min_subcorpora decades, computes the
# maximum and mean pairwise cosine distance across all decade positions.
# Words at the top of this ranking changed meaning the most over time.

def most_dispersed_words(aligned_to_global, topn=30, min_subcorpora=2):
    """
    Rank words by semantic drift across decades.

    Returns a DataFrame sorted by max_pairwise_dist (descending):
        word | n_subcorpora | max_pairwise_dist | mean_pairwise_dist | farthest_pair
    """
    word_to_decades = defaultdict(list)
    for decade, vecs in aligned_to_global.items():
        for word in vecs:
            word_to_decades[word].append(decade)

    rows = []
    for word, decades in word_to_decades.items():
        if len(decades) < min_subcorpora:
            continue

        dists     = []
        farthest  = (None, None, 0.0)

        for d1, d2 in combinations(sorted(decades), 2):
            v1   = aligned_to_global[d1][word]
            v2   = aligned_to_global[d2][word]
            dist = float(cosine_dist(v1, v2))
            dists.append(dist)
            if dist > farthest[2]:
                farthest = (d1, d2, dist)

        rows.append({
            "word":               word,
            "n_subcorpora":       len(decades),
            "max_pairwise_dist":  round(max(dists), 4),
            "mean_pairwise_dist": round(sum(dists) / len(dists), 4),
            "farthest_pair":      f"{farthest[0]} ↔ {farthest[1]}",
        })

    return (
        pd.DataFrame(rows)
        .sort_values("max_pairwise_dist", ascending=False)
        .reset_index(drop=True)
        .head(topn)
    )


## Interactive Plotly plot


In [ ]:
# Three layers drawn back-to-front:
#   Layer 1 — grey text labels  = global vocabulary background
#   Layer 2 — coloured dots     = decade positions on the trajectory line
#   Layer 3 — red X             = query word in the global model
#
# Plotly features available in the rendered plot:
#    Hover any dot        shows "love 1830s"
#    Hover any bg word    shows the word clearly
#    Draw a box           zooms into that region
#    Drag                pan around
#    Click legend item    hide/show that layer
#    Double-click         reset zoom

def plot_global_trajectory_interactive(
    query_word,
    aligned_to_global,
    global_model,
    pca_global,
    global_bg_coords,
    all_global_words,
    n_show = 2000,       # max background words to show (reduce if slow)
):
    """
    Interactive Plotly trajectory plot in the global PCA space.
    Returns the Plotly figure object and displays it.
    """
    positions = get_decade_positions(aligned_to_global, query_word, pca_global)
    if len(positions) < 2:
        print(f"  '{query_word}' found in fewer than 2 decades — "
              "cannot draw a trajectory.")
        return None

    fig = go.Figure()

    # Layer 1: background word cloud
    # mode="text" draws text labels without marker dots.
    # The labels are tiny (size=7) but become readable on zoom.
    # hoverinfo="text" shows the word in a tooltip on hover.
    fig.add_trace(go.Scatter(
        x         = global_bg_coords[:n_show, 0].tolist(),
        y         = global_bg_coords[:n_show, 1].tolist(),
        mode      = "text",
        text      = list(all_global_words[:n_show]),
        textfont  = dict(size=7, color="rgba(180,180,180,0.7)"),
        hovertext = list(all_global_words[:n_show]),
        hoverinfo = "text",
        name      = "Global vocabulary",
    ))

    # Layer 2: trajectory line + decade dots
    # The marker colour is set to the decade values and mapped through
    # dark purple = earliest, yellow = most recent.
    # showscale=True adds the colorbar legend on the right.
    xs      = [p[1] for p in positions]
    ys      = [p[2] for p in positions]
    decades = [p[0] for p in positions]

    fig.add_trace(go.Scatter(
        x    = xs,
        y    = ys,
        mode = "lines+markers+text",
        line = dict(color="rgba(0,0,0,0.45)", width=1.5),
        marker = dict(
            size       = 16,
            color      = decades,
            colorscale = "Viridis",
            showscale  = True,
            colorbar   = dict(
                title      = "Decade",
                x          = 1.02,
                tickvals   = [min(decades), max(decades)],
                ticktext   = [str(min(decades)), str(max(decades))],
            ),
            line = dict(color="black", width=1),
        ),
        text         = [str(d) for d in decades],
        textposition = "top right",
        textfont     = dict(size=10, color="black"),
        hovertext    = [f"{query_word} {d}s" for d in decades],
        hoverinfo    = "text",
        name         = f"Trajectory of '{query_word}'",
    ))

    # Layer 3: global model reference point — red X
    # Its position is the "temporal average" of the word across the corpus.
    # A decade dot near the red X = that decade's usage is close to the average.
    # A decade dot far from the red X = that decade diverged from the average.
    if query_word in global_model.wv:
        gvec = pca_global.transform(
            global_model.wv[query_word].reshape(1, -1)
        )[0]
        fig.add_trace(go.Scatter(
            x    = [float(gvec[0])],
            y    = [float(gvec[1])],
            mode = "markers+text",
            marker = dict(
                size   = 22,
                symbol = "x",
                color  = "red",
                line   = dict(color="darkred", width=2.5),
            ),
            text         = [f"{query_word} (global)"],
            textposition = "top right",
            textfont     = dict(size=11, color="red"),
            hovertext    = [f"{query_word} — global model (all decades combined)"],
            hoverinfo    = "text",
            name         = f"'{query_word}' global model",
        ))

    # Layout
    fig.update_layout(
        title = dict(
            text     = (
                f"Évolution sémantique de '<b>{query_word}</b>' "
                f"entre sous-corpus<br>"
                f"<sup>fond = vocabulaire global   "
                f"✕ = position modèle global</sup>"
            ),
            font     = dict(size=15),
            x        = 0.5,
            xanchor  = "center",
        ),
        xaxis_title = f"Axe 1 ({var1:.1f} %)",
        yaxis_title = f"Axe 2 ({var2:.1f} %)",
        plot_bgcolor = "rgba(235,240,248,0.6)",   # light blue-grey background
        paper_bgcolor = "white",
        width        = 1200,
        height       = 820,
        legend       = dict(
            x           = 1.10,
            y           = 1,
            bordercolor = "lightgrey",
            borderwidth = 1,
        ),
        margin = dict(l=60, r=180, t=80, b=60),
    )

    fig.show()
    return fig


## Run the plot once with the configured word


In [19]:
print(f"\nGenerating interactive plot for '{QUERY_WORD}' …")
plot_global_trajectory_interactive(
    query_word        = QUERY_WORD,
    aligned_to_global = aligned_to_global,
    global_model      = global_model,
    pca_global        = pca_global,
    global_bg_coords  = global_bg_coords,
    all_global_words  = all_global_words,
    n_show            = N_BACKGROUND,
)


# 16. INTERACTIVE WORD EXPLORER WIDGET
# Everything above (training, alignment, PCA) already ran and stays in memory.
# This widget re-runs only the display functions when you change the word —
# no re-training, no re-alignment, no re-fitting PCA.
#
# Controls:
#   Word box       — type any word, autocomplete from aligned vocabulary
#   Neighbours     — how many nearest neighbours to show in the table
#   Background N   — how many grey background words to show on the plot
#   Checkboxes     — toggle each output section on or off
#   Generate     — run everything for the chosen word
#
# Install if needed:
#   pip install ipywidgets
#   In Google Colab the next two lines are required:
#     from google.colab import output
#     output.enable_custom_widget_manager()

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
from IPython.display import display, clear_output

# Build the word list the user can choose from
# Only include words present in at least 2 aligned decades so every word
# produces an actual trajectory rather than a single isolated dot.

word_decade_counts = defaultdict(int)
for decade, vecs in aligned_to_global.items():
    for word in vecs:
        word_decade_counts[word] += 1

available_words = sorted(
    w for w, count in word_decade_counts.items() if count >= 2
)
print(f"Words available for exploration: {len(available_words)}")

# Widget controls

word_input = widgets.Combobox(
    value         = QUERY_WORD,
    options       = available_words,
    description   = "Word:",
    placeholder   = "type a word …",
    ensure_option = False,
    layout        = widgets.Layout(width="300px"),
    style         = {"description_width": "60px"},
)

topn_slider = widgets.IntSlider(
    value=10, min=3, max=25, step=1,
    description="Neighbours:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"),
)

bg_slider = widgets.IntSlider(
    value=N_BACKGROUND, min=500, max=5000, step=500,
    description="Background N:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="380px"),
)

# Checkboxes — toggle each output section
show_plot       = widgets.Checkbox(value=True,  description="Interactive plot")
show_distances  = widgets.Checkbox(value=True,  description="Distance table + bar chart")
show_neighbours = widgets.Checkbox(value=True,  description="Nearest neighbours table")
show_dispersed  = widgets.Checkbox(value=False, description="Most dispersed words (slow)")

run_btn = widgets.Button(
    description  = "▶  Generate",
    button_style = "primary",
    layout       = widgets.Layout(width="160px", height="36px"),
)

status = widgets.HTML(value="")
out    = widgets.Output()

#  Assemble the panel

panel = widgets.VBox([
    widgets.HTML(
        "<h3 style='margin:4px 0'>Word Explorer</h3>"
        "<p style='color:grey;margin:0'>Training is already done — "
        "only display reruns when you click Generate.</p>"
        "<hr style='margin:6px 0'>"
    ),
    widgets.HBox([word_input, run_btn, status]),
    widgets.HBox([topn_slider, bg_slider]),
    widgets.HTML("<b>Show:</b>"),
    widgets.HBox([show_plot, show_distances, show_neighbours, show_dispersed]),
])

display(panel, out)


# Button callback
def on_run(button):
    """Called when ▶ Generate is clicked. Only runs display functions."""
    word = word_input.value.strip().lower()
    topn = topn_slider.value
    n_bg = bg_slider.value

    # Validate input
    if not word:
        status.value = "<span style='color:red'>⚠ Please enter a word.</span>"
        return

    found_in = [d for d in aligned_to_global if word in aligned_to_global[d]]

    if not found_in:
        status.value = (
            f"<span style='color:red'>⚠ '{word}' not found in any decade. "
            "Try another word.</span>"
        )
        return

    if len(found_in) < 2:
        status.value = (
            f"<span style='color:orange'>⚠ '{word}' only in {found_in[0]}s — "
            "need ≥ 2 decades for a trajectory.</span>"
        )
        return

    status.value = f"<span style='color:#1a73e8'>Running for '{word}' …</span>"

    with out:
        clear_output(wait=True)

        print(f"{'═'*60}")
        print(f"  '{word}'   found in {len(found_in)} decades: "
              f"{[str(d)+'s' for d in sorted(found_in)]}")
        print(f"{'═'*60}\n")

        # Interactive Plotly plot
        if show_plot.value:
            plot_global_trajectory_interactive(
                query_word        = word,
                aligned_to_global = aligned_to_global,
                global_model      = global_model,
                pca_global        = pca_global,
                global_bg_coords  = global_bg_coords,
                all_global_words  = all_global_words,
                n_show            = n_bg,
            )

        # Distance table + bar chart
        # Cosine distance between consecutive decade positions.
        # Higher = bigger semantic shift in that step.
        if show_distances.value:
            print("── Distance per decade step ──────────────────────────")
            dist_df = compute_trajectory_distances(aligned_to_global, word)

            if dist_df.empty:
                print("  Not enough decades for distance calculation.")
            else:
                # Summary table
                print(f"\n{'Step':<22} {'Cosine':>8} {'Euclidean':>12}")
                print("─" * 46)
                for _, row in dist_df.iterrows():
                    print(f"{row['step']:<22} "
                          f"{row['cosine_dist']:>8.4f} "
                          f"{row['euclidean_dist']:>12.4f}")

                total_cos = dist_df['cosine_dist'].sum()
                total_euc = dist_df['euclidean_dist'].sum()
                print(f"\n  Total cosine distance:    {total_cos:.4f}")
                print(f"  Total euclidean distance: {total_euc:.4f}")

                big = dist_df.loc[dist_df['cosine_dist'].idxmax()]
                sml = dist_df.loc[dist_df['cosine_dist'].idxmin()]
                print(f"\n  Biggest shift:  {big['step']}  ({big['cosine_dist']:.4f})")
                print(f"  Smallest shift: {sml['step']}  ({sml['cosine_dist']:.4f})")

                # Bar chart — colour-coded red (big shift) → green (small shift)
                fig_bar, ax = plt.subplots(
                    figsize=(max(8, len(dist_df) * 1.2), 4)
                )
                bar_colors = [
                    plt.cm.RdYlGn_r(v / max(dist_df["cosine_dist"].max(), 1e-6))
                    for v in dist_df["cosine_dist"]
                ]
                bars = ax.bar(
                    dist_df["step"], dist_df["cosine_dist"],
                    color=bar_colors, edgecolor="black", linewidth=0.6,
                )
                for bar, val in zip(bars, dist_df["cosine_dist"]):
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.001,
                        f"{val:.3f}",
                        ha="center", va="bottom", fontsize=8,
                    )
                ax.set_title(
                    f"Cosine distance per step — '{word}'\n"
                    "(higher = bigger semantic shift)", fontsize=11,
                )
                ax.set_xlabel("Decade transition", fontsize=10)
                ax.set_ylabel("Cosine distance", fontsize=10)
                ax.set_xticklabels(
                    dist_df["step"], rotation=45, ha="right", fontsize=8
                )
                plt.subplots_adjust(left=0.1, right=0.95, top=0.88, bottom=0.25)
                plt.show()

        # Nearest neighbours table
        # Shows the topn most similar words per decade using aligned vectors.
        # Words that consistently appear across decades = stable neighbours.
        # Words that appear in only one decade = period-specific associations.
        if show_neighbours.value:
            print("\n── Nearest neighbours per sub-corpus ────────────────")
            nn = nearest_neighbours_per_decade(
                aligned_to_global=aligned_to_global,
                models=models,
                query_word=word,
                topn=topn,
                min_freq=1,
            )
            if not nn:
                print("  No data found.")
            else:
                n_cols      = 3
                decades_nn  = sorted(nn.keys())
                col_head    = f"{'word':<14} {'sim':>6} {'freq':>6} {'rank':>6}"

                for i in range(0, len(decades_nn), n_cols):
                    chunk = decades_nn[i: i + n_cols]
                    for d in chunk:
                        print(f"  Sous-corpus : {d}", end=" " * 18)
                    print()
                    for _ in chunk:
                        print("  " + "─" * 36, end=" " * 4)
                    print()
                    for _ in chunk:
                        print(f"  {col_head}", end=" " * 6)
                    print()
                    max_rows = max(len(nn[d]) for d in chunk)
                    for row_i in range(max_rows):
                        for d in chunk:
                            df = nn[d]
                            if row_i < len(df):
                                r    = df.iloc[row_i]
                                line = (f"  {r['word']:<14} "
                                        f"{r['similarity']:>6.4f} "
                                        f"{r['frequency']:>6} "
                                        f"{r['freq_rank']:>6}")
                            else:
                                line = ""
                            print(f"{line:<50}", end="")
                        print()
                    print()

        # Most dispersed words
        # Ranks ALL words by how far they traveled across decades.
        # Slow because it computes all pairwise distances — untick if not needed.
        if show_dispersed.value:
            print("\n── Most dispersed words (top 20) ────────────────────")
            disp = most_dispersed_words(
                aligned_to_global, topn=20, min_subcorpora=3
            )
            print(disp.to_string(index=False))

    status.value = (
        f"<span style='color:green'>✓ Done — "
        f"'{word}' across {len(found_in)} decades</span>"
    )


run_btn.on_click(on_run)

# Auto-run once immediately so there is output visible without clicking
on_run(None)



Generating interactive plot for 'love' …


Words available for exploration: 6115


Output()